# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

This notebook compares two retrieval pipelines over the same chunked PDF:

- **Pipeline 1 (baseline):** a Bi-Encoder embeds every chunk once. For a query, we take the Top 10 chunks by cosine similarity and pass the Top 3 to the LLM (Answer 1).
- **Pipeline 2 (re-ranking):** we take the same Top 10 candidates, score each (query, chunk) pair with a Cross-Encoder, re-sort, and pass the new Top 3 to the LLM (Answer 2).

Both pipelines use the same chunks, the same Bi-Encoder output for the first stage, the same prompt and the same LLM settings. The only thing that differs is which three chunks the LLM sees.

**Notebook layout**

1. Configuration
2. API key handling (encrypted)
3. Loading and chunking the PDF
4. Models and chunk embeddings
5. The two retrieval pipelines
6. LLM answer generation
7. Experiment helper
8. Test queries and failure-mode analysis
9. Rank shift table
10. Latency profiling and the scaling question
11. Report

## Installation manual

**Requirements:** Python 3.10 or newer, about 2 GB of free disk space for model downloads, and an internet connection for the first run (models are downloaded from Hugging Face and cached). A GPU is optional. The default models run on CPU as well, just more slowly.

**Steps**

1. Create and activate a virtual environment (recommended):
   - Windows: `python -m venv venv` then `venv\Scripts\activate`
   - Linux / macOS: `python3 -m venv venv` then `source venv/bin/activate`
2. Install the dependencies. Either run the cell below, or run `pip install -r requirements.txt` in a terminal.
3. If you have an NVIDIA GPU and want to use it, install the CUDA build of PyTorch first by following the selector at https://pytorch.org, then install the remaining packages. Check with `torch.cuda.is_available()` in the configuration cell.
4. Place your PDF next to this notebook and set `PDF_PATH` in the configuration cell.
5. Get an API key from OpenRouter (https://openrouter.ai/keys). Run `python setup_api_key.py` once in a terminal: it asks for the key (input is hidden) and stores it in encrypted form (see the API key section).
6. Run the cells from top to bottom.

In [1]:
# Dependencies already installed in environment
# %pip install -q pypdf sentence-transformers pandas numpy cryptography requests

## 1. Configuration

All the choices that affect the experiment live in this one cell, so they are easy to change and easy to report.

- **Bi-Encoder:** `BAAI/bge-base-en-v1.5` (110M parameters, 512 token input, about 440 MB).
- **Cross-Encoder:** `BAAI/bge-reranker-base` (278M parameters, 512 token input, about 1.1 GB).
- The two models together stay well under a 3 GB budget.
- To use the smaller models named in the assignment instead, set the Bi-Encoder to `all-MiniLM-L6-v2`, the Cross-Encoder to `cross-encoder/ms-marco-MiniLM-L-6-v2`, set `QUERY_PREFIX` to an empty string, and reduce `CHUNK_WORDS` to about 150 (that Bi-Encoder only reads 256 tokens).
- `QUERY_PREFIX` is a short instruction that the BGE models expect in front of **queries only**, never in front of chunks.
- Chunk size is in words. 200 words is roughly 260 to 300 tokens, comfortably inside the 512 token limit.

In [2]:
import os, re, time, textwrap, getpass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

# ---- Document ----
PDF_PATH = "GDPR.pdf"          # put your PDF next to the notebook

# ---- Models ----
BI_ENCODER_NAME    = "BAAI/bge-base-en-v1.5"
CROSS_ENCODER_NAME = "BAAI/bge-reranker-base"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # "" for MiniLM

# ---- Chunking ----
CHUNK_WORDS   = 200     # target chunk size in words
OVERLAP_WORDS = 40      # words repeated between neighbouring chunks

# ---- Retrieval ----
TOP_K = 10              # candidates retrieved by the Bi-Encoder
TOP_N = 3               # chunks given to the LLM

# ---- LLM ----
LLM_MODEL = "google/gemini-2.5-flash-lite"   # OpenRouter model id (cheap, non-reasoning)
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 2. API key handling

The OpenRouter API key should never be written in the notebook or committed to GitHub. This notebook stores it in encrypted form instead:

- `secret.key` holds a randomly generated encryption key.
- `api_key.enc` holds the API key, encrypted with that key (Fernet, from the `cryptography` package).
- Decryption only works when **both** files are present in the notebook folder.

On the first run, the cell asks for your API key (input is hidden) and offers to save the encrypted copy. On later runs it decrypts silently. If the files are missing, it falls back to the `OPENROUTER_API_KEY` environment variable, and then to a hidden prompt.

Please keep in mind what this does and does not protect. It stops the raw key from sitting in a readable file and means the `.enc` file alone is useless if it leaks. It does not protect against someone who has both files, so `secret.key` must stay out of Git. The included `.gitignore` already excludes both files.

In [3]:
from cryptography.fernet import Fernet, InvalidToken

KEY_FILE = Path("secret.key")
ENC_FILE = Path("api_key.enc")

def save_encrypted_key(api_key: str) -> None:
    """Create a fresh encryption key and store the API key encrypted with it."""
    fernet_key = Fernet.generate_key()
    KEY_FILE.write_bytes(fernet_key)
    ENC_FILE.write_bytes(Fernet(fernet_key).encrypt(api_key.encode()))
    print(f"Saved {ENC_FILE} and {KEY_FILE}. Do not commit either file.")

def get_api_key() -> str:
    # 1. Encrypted files from an earlier run
    if KEY_FILE.exists() and ENC_FILE.exists():
        try:
            return Fernet(KEY_FILE.read_bytes()).decrypt(ENC_FILE.read_bytes()).decode().strip()
        except InvalidToken:
            print("The key file does not match the encrypted file. Falling back to manual entry.")
    # 2. Environment variable
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if api_key:
        return api_key.strip()
    # 3. Interactive prompt fallback
    try:
        api_key = getpass.getpass("OpenRouter API key (hidden): ").strip()
        if api_key and input("Save an encrypted copy for next time? (y/n): ").strip().lower() == "y":
            save_encrypted_key(api_key)
        return api_key
    except Exception:
        print("Notice: No OPENROUTER_API_KEY set and interactive input unavailable. Set OPENROUTER_API_KEY or run setup_api_key.py.")
        return ""

API_KEY = get_api_key()
if API_KEY:
    print("API key loaded successfully.")
else:
    print("No API key loaded. Retrieval and re-ranking will execute; LLM answers require OPENROUTER_API_KEY.")

API key loaded successfully.


## 3. Loading and chunking the PDF

Chunking is plain text processing and involves no model. It decides what the retriever is able to find, so it affects every result downstream.

The approach used here:

1. Extract the text page by page and clean up hyphenated line breaks and extra spaces.
2. Split the text into sentences (and paragraph breaks).
3. Pack whole sentences into chunks of about `CHUNK_WORDS` words, never cutting a sentence in the middle.
4. Start each new chunk with the last `OVERLAP_WORDS` words of the previous one, so a rule and the sentence that follows it are less likely to be separated.

Each chunk keeps the page number it started on, which is useful when checking results against the PDF. A sentence that runs across a page break is split at the break, which is a small limitation of page-by-page extraction.

The same list of chunks is used by both pipelines.

In [4]:
def load_pages(path):
    reader = PdfReader(path)
    return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]

def clean(text):
    text = re.sub(r"-\n(\w)", r"\1", text)      # join words hyphenated across lines
    text = re.sub(r"[ \t]+", " ", text)           # collapse spaces and tabs
    return text

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+|\n\s*\n", text)
    return [re.sub(r"\s+", " ", p).strip() for p in parts if p.strip()]

def build_chunks(pages, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS):
    # Break the document into (sentence, page) units. A single very long sentence is
    # cut into pieces so it can never exceed the chunk size on its own.
    units = []
    for page_no, text in pages:
        for sentence in split_sentences(clean(text)):
            words = sentence.split()
            for j in range(0, len(words), chunk_words):
                units.append((" ".join(words[j:j + chunk_words]), page_no))

    groups, current, count = [], [], 0
    for sentence, page in units:
        n = len(sentence.split())
        if current and count + n > chunk_words:
            groups.append(current)
            # carry the tail of the finished chunk into the next one
            tail, t = [], 0
            for unit in reversed(current):
                tail.insert(0, unit)
                t += len(unit[0].split())
                if t >= overlap_words:
                    break
            current, count = tail, t
        current.append((sentence, page))
        count += n
    if current:
        groups.append(current)

    return [{"id": k, "page": g[0][1], "text": " ".join(s for s, _ in g)}
            for k, g in enumerate(groups)]

pages = load_pages(PDF_PATH)
chunks = build_chunks(pages)
chunk_texts = [c["text"] for c in chunks]

print(f"Pages: {len(pages)} | Chunks: {len(chunks)}")
print(f"Words per chunk: min {min(len(t.split()) for t in chunk_texts)}, "
      f"mean {np.mean([len(t.split()) for t in chunk_texts]):.0f}, "
      f"max {max(len(t.split()) for t in chunk_texts)}")
print("\nSample chunk (id 0):\n" + textwrap.fill(chunk_texts[0], 110))

Pages: 88 | Chunks: 537
Words per chunk: min 67, mean 189, max 436

Sample chunk (id 0):
I (Legislative acts) REGULATIONS REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 27
April 2016 on the protection of natural persons with regard to the processing of personal data and on the free
movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation) (Text with EEA
relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the
Functioning of the European Union, and in particular Article 16 thereof, Having regard to the proposal from
the European Commission, After transmission of the draft legislative act to the national parliaments, Having
regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the
Committee of the Regions (2), Acting in accordance with the ordinary legislative procedure (3), Whereas: (1)
The protection of natural persons in r

## 4. Models and chunk embeddings

Both models are transformers. They differ in what goes in and what comes out.

- **Bi-Encoder:** reads one text at a time and outputs a vector. Chunk vectors are computed once, here, and stored in a NumPy array. Vectors are normalized, so a plain dot product equals cosine similarity.
- **Cross-Encoder:** reads the query and one chunk together and outputs a single relevance score. Nothing can be precomputed for it.

The check after embedding counts tokens per chunk. Any chunk longer than the Bi-Encoder's limit would be silently truncated, so we want that number to be zero.

In [5]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE, max_length=512)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, normalize_embeddings=True,
                              batch_size=32, show_progress_bar=True)
print(f"Embedded {len(chunk_texts)} chunks in {time.perf_counter() - t0:.1f}s | "
      f"embedding matrix shape: {chunk_emb.shape}")

# Truncation check
token_counts = [len(bi_encoder.tokenizer.encode(t)) for t in chunk_texts]
limit = bi_encoder.max_seq_length
print(f"Bi-Encoder token limit: {limit} | longest chunk: {max(token_counts)} tokens | "
      f"chunks over the limit: {sum(n > limit for n in token_counts)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (514 > 512). Running this sequence through the model will result in indexing errors


Embedded 537 chunks in 86.9s | embedding matrix shape: (537, 768)


Bi-Encoder token limit: 512 | longest chunk: 542 tokens | chunks over the limit: 2


## 5. The two retrieval pipelines

`baseline_retrieve` is Pipeline 1, stage one. It embeds the query, takes the dot product with every chunk vector and returns the Top 10 chunk ids. Its first three results are the baseline context.

`rerank` is Pipeline 2, stage two. It builds 10 (query, chunk) pairs from those same candidates, scores them with the Cross-Encoder and returns them sorted by that score. Its first three results are the re-ranked context.

Both functions return their own elapsed time (measured with `time.perf_counter()`), which feeds the latency table later. Pipeline 2 needs the baseline step first, so its total cost is the sum of both.

In [6]:
def baseline_retrieve(query, k=TOP_K):
    """Pipeline 1: dense retrieval. Returns (chunk ids, cosine scores, seconds)."""
    t0 = time.perf_counter()
    q_vec = bi_encoder.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    scores = chunk_emb @ q_vec                      # cosine similarity (vectors are normalized)
    top = np.argsort(-scores)[:k]
    return top, scores[top], time.perf_counter() - t0

def rerank(query, candidate_ids):
    """Pipeline 2: Cross-Encoder re-ranking of the candidates.
    Returns (chunk ids sorted by Cross-Encoder score, scores, seconds)."""
    t0 = time.perf_counter()
    pairs = [(query, chunk_texts[i]) for i in candidate_ids]
    ce_scores = np.asarray(cross_encoder.predict(pairs, batch_size=len(pairs)))
    order = np.argsort(-ce_scores)
    return np.asarray(candidate_ids)[order], ce_scores[order], time.perf_counter() - t0

## 6. LLM answer generation

Both answers come from the same LLM (`google/gemini-2.5-flash-lite`, called through OpenRouter), the same prompt template and temperature 0, so any difference between Answer 1 and Answer 2 comes from the retrieved chunks and not from randomness or wording.

The prompt tells the model to use only the supplied context and to say "Not found in context." otherwise. This makes a missing chunk visible: the answer becomes wrong, incomplete or a refusal, instead of being quietly filled in from the model's general knowledge.

If the two pipelines return exactly the same three chunks in the same order, the second API call is skipped and the first answer is reused.

**Why OpenRouter.** The first runs used the Gemini API free tier directly, which allows only 5 requests per minute and 20 per day for this model. The daily limit ran out before the final run, so the final run calls Gemini Flash-Lite through OpenRouter instead. Calls are still spaced a little apart, and a 429 error makes the call wait and retry.

**Proof log and cache.** Every call is recorded with the exact prompt that was sent and the exact response that came back, and saved immediately to `llm_prompts_and_responses.json` (a readable copy, `llm_prompts_and_responses.md`, is written after the queries). If the notebook is run again, a call whose model and prompt are already in that file is not sent again; the saved response is reused. This keeps the answers in the report and the proof file identical, and avoids paying for the same call twice.

In [7]:
import json, requests

PROMPT_TEMPLATE = """You are answering a question about a document. Use ONLY the context below.
If the context does not contain enough information to answer, reply exactly: "Not found in context."
Do not use outside knowledge. Answer in at most 120 words, and state any exception or restriction that appears in the context.

Context:
{context}

Question: {question}

Answer:"""

LOG_FILE = Path("llm_prompts_and_responses.json")
MIN_SECONDS_BETWEEN_CALLS = 2

# Exact prompt and response of every call. Loaded from disk so a re-run reuses earlier calls.
LLM_LOG = json.loads(LOG_FILE.read_text()) if LOG_FILE.exists() else []
_last_call = [0.0]

def call_llm(prompt):
    """One chat completion request to OpenRouter, temperature 0."""
    r = requests.post(
        OPENROUTER_URL,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={"model": LLM_MODEL, "temperature": 0,
              "messages": [{"role": "user", "content": prompt}]},
        timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
    data = r.json()
    return (data["choices"][0]["message"]["content"] or "").strip(), data.get("model", LLM_MODEL)

def generate_answer(query, chunk_ids, pipeline, retries=3):
    if not API_KEY:
        return "[LLM skipped: no API key. Set OPENROUTER_API_KEY or run setup_api_key.py.]"
    context = "\n\n".join(f"[Chunk {i}, page {chunks[i]['page']}]\n{chunk_texts[i]}" for i in chunk_ids)
    prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    # reuse a saved response for the exact same model and prompt
    for e in LLM_LOG:
        if e["model"] == LLM_MODEL and e["prompt"] == prompt:
            return e["response"]

    for attempt in range(retries):
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        try:
            answer, served_by = call_llm(prompt)
            LLM_LOG.append(dict(query=query, pipeline=pipeline,
                                chunk_ids=[int(i) for i in chunk_ids],
                                model=LLM_MODEL, served_model=served_by, temperature=0,
                                timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
                                prompt=prompt, response=answer))
            LOG_FILE.write_text(json.dumps(LLM_LOG, indent=2, ensure_ascii=False))
            return answer
        except Exception as e:
            if attempt == retries - 1:
                raise
            delay = 30 if "429" in str(e) else 5 * (attempt + 1)
            print(f"LLM call failed ({str(e)[:150]}). Retrying in {delay}s...")
            time.sleep(delay)

## 7. Experiment helper

`run_query` runs one query through both pipelines and records everything needed for the report.

To measure rank shifts we need to know which chunk really contains the answer (the "gold" chunk). Rather than hard-coding chunk ids, you give a short, distinctive `gold_phrase` copied from the PDF text, and the helper finds the chunk(s) containing it. If the phrase is not found, it tells you, so you can fix it.

For each query it prints the baseline Top 10, the re-ranked order, the gold chunk's rank before and after, and both LLM answers.

In [8]:
def norm(s):
    return re.sub(r"\s+", " ", s).strip().lower()

def find_gold_chunks(gold_phrase):
    if not gold_phrase:
        return []
    phrase = norm(gold_phrase)
    return [c["id"] for c in chunks if phrase in norm(c["text"])]

def best_rank(order, gold_ids):
    order = [int(i) for i in order]
    ranks = [order.index(g) + 1 for g in gold_ids if g in order]
    return min(ranks) if ranks else None

def run_query(label, kind, query, gold_phrase="", call_llm=True, verbose=True):
    cand_ids, bi_scores, t_bi = baseline_retrieve(query)
    rr_ids, rr_scores, t_rr = rerank(query, cand_ids)

    base_top = [int(i) for i in cand_ids[:TOP_N]]
    rr_top = [int(i) for i in rr_ids[:TOP_N]]
    gold_ids = find_gold_chunks(gold_phrase)
    if gold_phrase and not gold_ids:
        print(f"Warning: gold phrase not found in any chunk for query '{label}'.")

    answer1 = answer2 = None
    if call_llm:
        answer1 = generate_answer(query, base_top, pipeline="Pipeline 1 (baseline)")
        answer2 = answer1 if rr_top == base_top else generate_answer(query, rr_top, pipeline="Pipeline 2 (re-ranked)")

    rec = dict(label=label, kind=kind, query=query, gold_ids=gold_ids,
               baseline_order=[int(i) for i in cand_ids], rerank_order=[int(i) for i in rr_ids],
               base_top=base_top, rr_top=rr_top,
               gold_rank_baseline=best_rank(cand_ids, gold_ids),
               gold_rank_rerank=best_rank(rr_ids, gold_ids),
               t_baseline=t_bi, t_rerank=t_rr, answer1=answer1, answer2=answer2)

    if verbose:
        print("=" * 100)
        print(f"[{kind}] {label}\nQuery: {query}")
        print(f"Gold chunk(s): {gold_ids if gold_ids else 'not specified'}")
        print(f"Baseline order (chunk ids):  {rec['baseline_order']}")
        print(f"Re-ranked order (chunk ids): {rec['rerank_order']}")
        print(f"Gold rank: baseline {rec['gold_rank_baseline']} -> re-ranked {rec['gold_rank_rerank']}")
        print(f"Top {TOP_N} given to LLM  | baseline: {base_top} | re-ranked: {rr_top}")
        print(f"Time: baseline retrieval {t_bi*1000:.1f} ms | re-ranking {t_rr*1000:.1f} ms")
        if call_llm:
            print("\nAnswer 1 (baseline):\n" + textwrap.fill(answer1, 100))
            print("\nAnswer 2 (re-ranked):\n" + textwrap.fill(answer2, 100))
    return rec

## 8. Test queries and failure-mode analysis

The queries were built by finding the passage in the PDF first and writing the question second, so the gold chunk is known in advance. Candidate questions were first checked with retrieval and re-ranking only (no LLM calls), and the ones below were kept.

| Type | What it tests |
|---|---|
| exception | The baseline finds the general rule but misses the "shall not be required if…" condition |
| negation | "Allowed" and "not allowed" passages look alike to the Bi-Encoder |
| keyword trap | The question shares words ("penalties") with general passages, while the answer uses different wording ("administrative fines up to 20 000 000 EUR") |
| buried chunk | The chunk with the actual list of cases sits at rank 4–10 behind recitals on the same topic |
| re-rank regression | A case where the baseline already worked and re-ranking made the context worse, kept as an honest negative result |
| exception (retention) | Same retention topic as the regression case, but here the baseline misses the "shall not apply" exception to erasure and re-ranking finds it |
| control | An easy question where both pipelines should agree |

Each entry has a `label`, a `kind`, the `query`, and a `gold_phrase` copied from the passage that contains the answer.

The first lines below warm up both models so that the first timed query is not slowed by one-off initialization.

In [9]:
# Warm-up so that the first timed query is not inflated by one-off initialization
_ids, _, _ = baseline_retrieve("warm up")
rerank("warm up", _ids)

TEST_QUERIES = [
    dict(
        label="Breach communication encryption exemption",
        kind="exception",
        query="When is a controller excused from communicating a personal data breach to the affected individuals because of encryption?",
        gold_phrase="unintelligible to any person who is not authorised to access it, such as encryption"
    ),
    dict(
        label="Direct marketing objection override prohibition",
        kind="negation",
        query="Can a company continue sending marketing emails by claiming legitimate interest if a customer objects?",
        gold_phrase="where the data subject objects to processing for direct marketing purposes, the personal data shall no longer be processed for such purposes"
    ),
    dict(
        label="Maximum administrative fine",
        kind="keyword trap",
        query="What penalties can a company face for violating the regulation?",
        gold_phrase="up to 20 000 000 EUR, or in the case of an undertaking, up to 4 %"
    ),
    dict(
        label="When a DPIA is mandatory",
        kind="buried chunk",
        query="When must a controller carry out a data protection impact assessment?",
        gold_phrase="systematic monitoring of a publicly accessible area on a large scale"
    ),
    dict(
        label="Storage limitation archiving exception",
        kind="re-rank regression",
        query="In what circumstances can personal data be retained longer than originally required?",
        gold_phrase="insofar as the personal data will be processed solely for archiving purposes in the public interest, scientific or historical research purposes or statistical purposes"
    ),
    dict(
        label="Erasure refused because of a legal obligation",
        kind="exception (retention)",
        query="Can a controller refuse an erasure request because a law obliges it to keep the data?",
        gold_phrase="for compliance with a legal obligation which requires processing by Union or Member State law"
    ),
    dict(
        label="72-hour breach notification deadline",
        kind="control",
        query="What is the maximum time limit for notifying the competent supervisory authority after becoming aware of a personal data breach?",
        gold_phrase="not later than 72 hours after having become aware of it"
    ),
]

results = [run_query(**q) for q in TEST_QUERIES]

[exception] Breach communication encryption exemption
Query: When is a controller excused from communicating a personal data breach to the affected individuals because of encryption?
Gold chunk(s): [350, 351]
Baseline order (chunk ids):  [123, 122, 124, 347, 350, 351, 349, 348, 121, 78]
Re-ranked order (chunk ids): [351, 122, 350, 347, 349, 123, 348, 124, 121, 78]
Gold rank: baseline 5 -> re-ranked 1
Top 3 given to LLM  | baseline: [123, 122, 124] | re-ranked: [351, 122, 350]
Time: baseline retrieval 60.2 ms | re-ranking 1682.1 ms

Answer 1 (baseline):
Not found in context.

Answer 2 (re-ranked):
A controller is excused from communicating a personal data breach to the affected individuals if
they have implemented appropriate technical and organizational protection measures, such as
encryption, that render the personal data unintelligible to unauthorized persons. This is one of the
conditions under which communication to the data subject is not required, as stated in paragraph
3(a) of t

[negation] Direct marketing objection override prohibition
Query: Can a company continue sending marketing emails by claiming legitimate interest if a customer objects?
Gold chunk(s): [314]
Baseline order (chunk ids):  [56, 57, 55, 314, 91, 92, 423, 189, 54, 280]
Re-ranked order (chunk ids): [57, 91, 314, 56, 55, 54, 423, 92, 189, 280]
Gold rank: baseline 4 -> re-ranked 3
Top 3 given to LLM  | baseline: [56, 57, 55] | re-ranked: [57, 91, 314]
Time: baseline retrieval 60.5 ms | re-ranking 1559.7 ms

Answer 1 (baseline):
The processing of personal data for direct marketing purposes may be regarded as carried out for a
legitimate interest. However, the interests and fundamental rights of the data subject could
override the interest of the data controller where personal data are processed in circumstances
where data subjects do not reasonably expect further processing. The existence of a legitimate
interest requires careful assessment, including whether a data subject can reasonably expect

[keyword trap] Maximum administrative fine
Query: What penalties can a company face for violating the regulation?
Gold chunk(s): [513, 514, 515, 516]
Baseline order (chunk ids):  [202, 511, 512, 513, 203, 514, 515, 204, 516, 201]
Re-ranked order (chunk ids): [514, 515, 511, 512, 516, 513, 201, 202, 204, 203]
Gold rank: baseline 4 -> re-ranked 1
Top 3 given to LLM  | baseline: [202, 511, 512] | re-ranked: [514, 515, 511]
Time: baseline retrieval 53.8 ms | re-ranking 2333.1 ms

Answer 1 (baseline):
Penalties for infringing the regulation can include administrative fines, or in the case of a minor
infringement or if the fine would be a disproportionate burden to a natural person, a reprimand may
be issued instead of a fine. The total amount of an administrative fine for intentional or negligent
infringements of multiple provisions for the same or linked processing operations shall not exceed
the amount specified for the gravest infringement. Infringements of specific provisions can result

[buried chunk] When a DPIA is mandatory
Query: When must a controller carry out a data protection impact assessment?
Gold chunk(s): [353, 354]
Baseline order (chunk ids):  [120, 128, 352, 121, 134, 353, 132, 359, 131, 358]
Re-ranked order (chunk ids): [352, 353, 128, 120, 359, 132, 131, 358, 121, 134]
Gold rank: baseline 6 -> re-ranked 2
Top 3 given to LLM  | baseline: [120, 128, 352] | re-ranked: [352, 353, 128]
Time: baseline retrieval 61.6 ms | re-ranking 1622.7 ms

Answer 1 (baseline):
A controller must carry out a data protection impact assessment prior to processing operations that
are likely to result in a high risk to the rights and freedoms of natural persons. This is
particularly relevant when using new technologies, processing a considerable amount of personal data
at a large scale, or when processing operations make it more difficult for data subjects to exercise
their rights.

Answer 2 (re-ranked):
A controller must carry out a data protection impact assessment prior to pr

[re-rank regression] Storage limitation archiving exception
Query: In what circumstances can personal data be retained longer than originally required?
Gold chunk(s): [253, 254, 255]
Baseline order (chunk ids):  [254, 255, 84, 45, 85, 79, 295, 78, 122, 217]
Re-ranked order (chunk ids): [254, 255, 78, 45, 84, 85, 295, 79, 122, 217]
Gold rank: baseline 1 -> re-ranked 1
Top 3 given to LLM  | baseline: [254, 255, 84] | re-ranked: [254, 255, 78]
Time: baseline retrieval 52.1 ms | re-ranking 3084.1 ms

Answer 1 (baseline):
Personal data may be stored for longer periods if processed solely for archiving purposes in the
public interest, scientific or historical research purposes, or statistical purposes, in accordance
with Article 89(1). This is subject to the implementation of appropriate technical and
organizational measures to safeguard data subject rights and freedoms. Additionally, further
retention is lawful for exercising the right of freedom of expression and information, compliance
wi

[exception (retention)] Erasure refused because of a legal obligation
Query: Can a controller refuse an erasure request because a law obliges it to keep the data?
Gold chunk(s): [306, 307, 308]
Baseline order (chunk ids):  [304, 90, 89, 86, 85, 305, 82, 309, 306, 303]
Re-ranked order (chunk ids): [304, 306, 309, 305, 303, 85, 82, 86, 89, 90]
Gold rank: baseline 9 -> re-ranked 2
Top 3 given to LLM  | baseline: [304, 90, 89] | re-ranked: [304, 306, 309]
Time: baseline retrieval 64.1 ms | re-ranking 1812.0 ms

Answer 1 (baseline):
Yes, a controller can refuse an erasure request if the personal data has to be erased for compliance
with a legal obligation in Union or Member State law to which the controller is subject. This is one
of the grounds for erasure, but its absence would allow the controller to retain the data.

Answer 2 (re-ranked):
Yes, a controller can refuse an erasure request if processing is necessary for compliance with a
legal obligation which requires processing by Union o

[control] 72-hour breach notification deadline
Query: What is the maximum time limit for notifying the competent supervisory authority after becoming aware of a personal data breach?
Gold chunk(s): [122, 123, 346, 347]
Baseline order (chunk ids):  [347, 123, 348, 122, 125, 346, 124, 349, 121, 361]
Re-ranked order (chunk ids): [347, 346, 348, 123, 122, 125, 124, 121, 349, 361]
Gold rank: baseline 1 -> re-ranked 1
Top 3 given to LLM  | baseline: [347, 123, 348] | re-ranked: [347, 346, 348]
Time: baseline retrieval 66.7 ms | re-ranking 1758.3 ms

Answer 1 (baseline):
The controller shall notify the personal data breach to the competent supervisory authority without
undue delay and, where feasible, not later than 72 hours after having become aware of it, unless the
personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons.
Where the notification is not made within 72 hours, it shall be accompanied by reasons for the
delay.

Answer 2 (re-ranked):
The

### Saving the exact prompts and responses

Every LLM call made above is in two files, so the answers quoted in the report can be checked:

- `llm_prompts_and_responses.md`: readable version, one section per call, with the full prompt (including the chunk text) and the raw response.
- `llm_prompts_and_responses.json`: the same data in machine-readable form.

In [10]:
import json


lines = ["# LLM prompts and responses", "",
         f"Model: `{LLM_MODEL}` via OpenRouter | temperature 0", ""]
for r in results:
    calls = [e for e in LLM_LOG if e["query"] == r["query"] and e["model"] == LLM_MODEL]
    lines += [f"## [{r['kind']}] {r['label']}", "", f"**Query:** {r['query']}", ""]
    if r["rr_top"] == r["base_top"]:
        lines += ["Both pipelines picked the same top 3 chunks, so one call was made and the answer was reused.", ""]
    for e in calls:
        lines += [f"### {e['pipeline']}: chunks {e['chunk_ids']} ({e['timestamp']})", "",
                  "**Prompt sent:**", "", "```text", e["prompt"], "```", "",
                  "**Response received:**", "", "```text", e["response"], "```", ""]

with open("llm_prompts_and_responses.md", "w") as f:
    f.write("\n".join(lines))
print("Saved llm_prompts_and_responses.md (all prompts and responses are also in llm_prompts_and_responses.json)")

Saved llm_prompts_and_responses.md (all prompts and responses are also in llm_prompts_and_responses.json)


## 9. Rank shift table

This table summarises, for every query, where the gold chunk sat in the baseline ranking and where it ends up after re-ranking. "Not in top 10" means the Bi-Encoder never surfaced it, in which case re-ranking cannot help, because the Cross-Encoder only sees the 10 candidates.

In [11]:
def fmt(rank):
    return f"#{rank}" if rank else "not in top 10"

rank_table = pd.DataFrame([{
    "Query": r["label"],
    "Type": r["kind"],
    "Gold chunk": ", ".join(map(str, r["gold_ids"])) or "-",
    "Baseline rank": fmt(r["gold_rank_baseline"]),
    "Re-ranked rank": fmt(r["gold_rank_rerank"]),
    "In baseline top 3": bool(r["gold_rank_baseline"] and r["gold_rank_baseline"] <= TOP_N),
    "In re-ranked top 3": bool(r["gold_rank_rerank"] and r["gold_rank_rerank"] <= TOP_N),
} for r in results])

# Export rank shift table for persistent evaluation and reporting
rank_table.to_csv("rank_shift_table.csv", index=False)
print("✓ Saved rank shift table to file: rank_shift_table.csv")
rank_table


✓ Saved rank shift table to file: rank_shift_table.csv


,Query,Type,Gold chunk,Baseline rank,Re-ranked rank,In baseline top 3,In re-ranked top 3
0,Breach communication encryption exemption,exception,"350, 351",#5,#1,False,True
1,Direct marketing objection override prohibition,negation,314,#4,#3,False,True
2,Maximum administrative fine,keyword trap,"513, 514, 515, 516",#4,#1,False,True
3,When a DPIA is mandatory,buried chunk,"353, 354",#6,#2,False,True
4,Storage limitation archiving exception,re-rank regression,"253, 254, 255",#1,#1,True,True
5,Erasure refused because of a legal obligation,exception (retention),"306, 307, 308",#9,#2,False,True
6,72-hour breach notification deadline,control,"122, 123, 346, 347",#1,#1,True,True


## 10. Latency profiling and the scaling question

The timings recorded during the query runs above are single measurements and can be noisy on CPU. For the latency table, each query is therefore run 10 more times through both stages (no LLM calls), and the table summarises all of those runs. Pipeline 2 reuses the baseline retrieval and then adds the re-ranking step, so its total is the sum of the two. LLM time is not included because it depends on the API and is the same for both pipelines.

The second cell answers the scaling question with a measurement. It runs the Cross-Encoder on **every** chunk for one query, which is what we would have to do if there was no Bi-Encoder stage, and extrapolates the time to a collection of 100,000 chunks. This is a linear estimate, since every (query, chunk) pair needs its own forward pass.

In [12]:
N_REPEATS = 10
t_base, t_rr = [], []
for q in TEST_QUERIES:
    for _ in range(N_REPEATS):
        ids, _, t1 = baseline_retrieve(q["query"])
        _, _, t2 = rerank(q["query"], ids)
        t_base.append(t1)
        t_rr.append(t2)

t_base = np.array(t_base) * 1000
t_rr   = np.array(t_rr) * 1000
t_p2   = t_base + t_rr

latency_table = pd.DataFrame({
    "Stage": ["Pipeline 1: query embedding + cosine search (Top 10)",
              "Re-ranking: Cross-Encoder on 10 pairs",
              "Pipeline 2 total: retrieval + re-ranking"],
    "Median (ms)": [np.median(t_base), np.median(t_rr), np.median(t_p2)],
    "Mean (ms)":   [t_base.mean(), t_rr.mean(), t_p2.mean()],
    "Min (ms)":    [t_base.min(),  t_rr.min(),  t_p2.min()],
    "Max (ms)":    [t_base.max(),  t_rr.max(),  t_p2.max()],
}).round(1)

latency_table.to_csv("latency_table.csv", index=False)
print(f"✓ Saved latency table ({len(TEST_QUERIES)} queries x {N_REPEATS} runs) to latency_table.csv")

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("✓ Saved full evaluation results to evaluation_results.json")

display(latency_table)
print(f"Pipeline 2 is about {np.median(t_p2) / np.median(t_base):.1f}x slower than Pipeline 1 (median, {DEVICE}).")

✓ Saved latency table (7 queries x 10 runs) to latency_table.csv
✓ Saved full evaluation results to evaluation_results.json


,Stage,Median (ms),Mean (ms),Min (ms),Max (ms)
0,Pipeline 1: query embedding + cosine search (T...,56.2,57.3,49.9,75.6
1,Re-ranking: Cross-Encoder on 10 pairs,1760.5,1991.1,1534.0,3136.8
2,Pipeline 2 total: retrieval + re-ranking,1824.2,2048.4,1588.8,3191.1


Pipeline 2 is about 32.5x slower than Pipeline 1 (median, cpu).


In [13]:
# Scaling experiment: Cross-Encoder over the whole collection for one query
q = TEST_QUERIES[0]["query"]
t0 = time.perf_counter()
cross_encoder.predict([(q, t) for t in chunk_texts], batch_size=32)
t_all = time.perf_counter() - t0

per_pair = t_all / len(chunk_texts)
print(f"Cross-Encoder over all {len(chunk_texts)} chunks: {t_all:.2f}s ({per_pair * 1000:.1f} ms per pair)")
print(f"Estimated for 100,000 chunks: {per_pair * 100_000 / 60:.1f} minutes for a single query")
print(f"Baseline retrieval for the same query: {results[0]['t_baseline'] * 1000:.1f} ms")

Cross-Encoder over all 537 chunks: 107.20s (199.6 ms per pair)
Estimated for 100,000 chunks: 332.7 minutes for a single query
Baseline retrieval for the same query: 60.2 ms


## 11. Report

The written report, with the architecture comparison, rank shift analysis, latency table and scaling answer, is in `EVALUATION_REPORT.md`. The exact prompts and responses behind every answer quoted there are in `llm_prompts_and_responses.md`.